# Generation Metrics

Faithfullness - Meadure for hallucination

This metric assesses the factual accuracy of the generated answers by checking if the statements made in the answers are supported by the provided context.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import asyncio
from ragas.llms import LangchainLLMWrapper
from ragas import SingleTurnSample 
from ragas.metrics import Faithfulness
from langchain_groq import ChatGroq

c:\Users\ASUS\.conda\envs\graph-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [4]:
ragas_llm = LangchainLLMWrapper(llm)

In [5]:
from ragas import SingleTurnSample 
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = Faithfulness(llm=ragas_llm)
score = await scorer.single_turn_ascore(sample)
print(score)

1.0


Answer Relevancy :This metric measures how relevant and directly related the generated answer is to the posed question. 

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [7]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27580\2131500445.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [8]:
from ragas.metrics import ResponseRelevancy
from ragas.embeddings import LangchainEmbeddingsWrapper

In [9]:
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

In [10]:
sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
score = await scorer.single_turn_ascore(sample)
print(score)

0.8993190750083979


# Retrieval Metrics

Context Recall - how many right information is actually present in the retrieved docs (context)

In [11]:
from ragas.metrics import ContextRecall

reference == gronnd truth

In [12]:
sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["Paris is the capital of France."], 
)
context_recall = ContextRecall(llm=ragas_llm)
score = await context_recall.single_turn_ascore(sample)
print(score)

1.0


Context Precision - evaluates the retriever's ability to rank relevant chunks higher than irrelevant ones for a given query in the retrieved context, conveying the quality of the retrieval pipeline.

In [13]:
from ragas.metrics import ContextPrecision

In [14]:
sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."], 
)
context_precision = ContextPrecision(llm=ragas_llm)
score = await context_precision.single_turn_ascore(sample)  
print(score)

0.9999999999


### Generating Ground Truth (reference) using TestSetGenerator

automatically generates questions from your documents

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [16]:
docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(
    PyPDFLoader("attention.pdf").load()
)

testset_size = total number of Q&A test cases generated

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=ragas_llm, embedding_model=ragas_embeddings)
testset = generator.generate_with_langchain_docs(docs, testset_size=5)

In [ ]:
for row in testset.to_pandas().itertuples():
    run(row.user_input, ground_truth=row.reference)

OR

In [ ]:
for row in testset_df.itertuples():
    answer, retrieved = ask(row.user_input)
    eval_buffer.append(SingleTurnSample(
        user_input=row.user_input,
        response=answer,
        retrieved_contexts=[d.page_content for d in retrieved],
        reference=row.reference   # ground truth from generator → enables context_recall
    ))

In [ ]:
run_eval()